# SentencePiece: A Language-Independent Subword Tokenizer

## What is SentencePiece?

**SentencePiece** is a language-independent subword tokenization and detokenization algorithm developed by Google. It treats the input text as a raw sequence of Unicode characters and uses a unsupervised training approach to automatically discover optimal subword vocabularies.

**Paper**: [SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing](https://arxiv.org/abs/1808.06226) - Kudo & Richardson, 2018

## Key Features of SentencePiece

1. **Language-Independent**: No language-specific preprocessing required
2. **Direct Training on Raw Text**: Treats text as a sequence of Unicode characters
3. **Reversible Tokenization**: Can reconstruct original text from tokenized sequence
4. **Built-in Subword Sampling**: Supports subword regularization for better generalization
5. **NFKC Normalization**: Applies Unicode normalization for consistency

## How SentencePiece Works

SentencePiece uses one of two algorithms:

### 1. Unigram Language Model (Default)
- Probabilistic model that selects optimal subword vocabulary
- Uses Viterbi algorithm to find the best segmentation
- Most commonly used approach

### 2. Byte Pair Encoding (BPE)
- Same algorithm as in subword-nmt
- Merges frequent adjacent character pairs

### Process Flow:
```
Raw Text → Normalization → Pre-tokenization (optional) → Training → Vocabulary Building → Tokenization
```

## Installation

In [ ]:
# Install SentencePiece
!pip install sentencepiece

## Training SentencePiece Model

In [ ]:
import sentencepiece as spm
import os

corpus = """
Natural language processing is a subfield of linguistics, computer science, and artificial intelligence.
It is concerned with the interactions between computers and human language.
NLP enables computers to understand, interpret, and generate human language in a valuable way.
Deep learning has revolutionized the field of natural language processing.
Transformers have become the dominant architecture for NLP tasks.
BERT and GPT are famous language models that use subword tokenization.
Tokenization is the process of converting text into tokens.
Subword tokenization breaks words into smaller units to handle rare words.
"""

# Save corpus to file
with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(corpus)

print("Corpus saved to corpus.txt")

In [ ]:
# Train SentencePiece model with Unigram LM
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='sp_model',
    vocab_size=100,  # Vocabulary size
    character_coverage=1.0,  # Coverage of characters (1.0 = 100%)
    model_type='unigram',  # Model type: 'unigram', 'bpe', 'char', 'word'
    max_sentence_length=8192,  # Max sentence length
    pad_id=0,  # Padding token ID
    unk_id=1,  # Unknown token ID
    bos_id=2,  # Beginning of sentence token ID
    eos_id=3,  # End of sentence token ID
    pad_piece='[PAD]',
    unk_piece='[UNK]',
    bos_piece='[BOS]',
    eos_piece='[EOS]'
)

print("Model trained and saved!")

## Loading and Using the Model

In [ ]:
# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load('sp_model.model')

print("Model loaded successfully!")
print(f"Vocabulary size: {sp.get_piece_size()}")

In [ ]:
# Encode: Text to IDs
text = "Natural language processing is amazing"
pieces = sp.encode(text, out_type='str')
ids = sp.encode(text, out_type='int')

print(f"Original text: {text}")
print(f"Tokenized (pieces): {pieces}")
print(f"Tokenized (IDs): {ids}")

In [ ]:
# Decode: IDs back to Text
decoded_text = sp.decode(ids)
decoded_pieces = sp.decode(pieces)

print(f"Decoded from IDs: {decoded_text}")
print(f"Decoded from pieces: {decoded_pieces}")

## Advanced Features

In [ ]:
# Batch encoding
texts = [
    "Deep learning transforms NLP.",
    "Transformers use attention mechanisms.",
    "BERT stands for Bidirectional Encoder Representations."
]

batch_ids = sp.encode(texts, out_type='int')
batch_pieces = sp.encode(texts, out_type='str')

for i, text in enumerate(texts):
    print(f"Text: {text}")
    print(f"IDs: {batch_ids[i]}")
    print(f"Pieces: {batch_pieces[i]}")
    print()

In [ ]:
# Vocabulary inspection
print("Sample vocabulary (first 20 pieces):")
for i in range(min(20, sp.get_piece_size())):
    piece = sp.id_to_piece(i)
    print(f"  ID {i}: {repr(piece)}")

In [ ]:
# Get piece probability (for unigram model)
piece = "▁NLP"  # underscore prefix indicates start of word
prob = sp.get_score(piece)
print(f"Score for '{piece}': {prob}")

# Get piece probability for all common pieces
print("\nTop 10 pieces by score:")
pieces_with_scores = [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())]
sorted_pieces = sorted(pieces_with_scores, key=lambda x: x[1], reverse=True)
for piece, score in sorted_pieces[:10]:
    print(f"  {repr(piece)}: {score}")

## Comparison: BPE vs Unigram

In [ ]:
# Train both BPE and Unigram models for comparison
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='sp_bpe',
    vocab_size=100,
    model_type='bpe'
)

spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='sp_unigram',
    vocab_size=100,
    model_type='unigram'
)

sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load('sp_bpe.model')

sp_unigram = spm.SentencePieceProcessor()
sp_unigram.load('sp_unigram.model')

test_text = "Natural language processing handles rare words elegantly."

print("Comparison on text:", test_text)
print()
print(f"BPE tokens:      {sp_bpe.encode(test_text, out_type='str')}")
print(f"Unigram tokens:  {sp_unigram.encode(test_text, out_type='str')}")

## SentencePiece in Popular Models

| Model | Tokenizer | Notes |
|-------|-----------|-------|
| **ALBERT** | SentencePiece | Language-agnostic training |
| **T5** | SentencePiece | Uses Unigram model |
| **XLNet** | SentencePiece | Handles long sequences |
| **mBART** | SentencePiece | Multilingual BART |
| **GPT-2** | BPE | Uses byte-level BPE |
| **BERT** | WordPiece | Language-specific approach |

## Advantages of SentencePiece

1. **No Pre-tokenization Required**
   - Doesn't need language-specific tokenizers (spaces, punctuation rules)
   - Works directly on raw Unicode text

2. **Reversible Tokenization**
   - Can reconstruct exact original text from tokens
   - Essential for tasks like speech recognition

3. **Language Independence**
   - Single tokenizer for multiple languages
   - Especially useful for multilingual models

4. **Handles Any Character Set**
   - Unicode-native
   - No out-of-vocabulary characters

5. **Subword Regularization**
   - Supports multiple segmentations during training
   - Improves model generalization

## Limitations of SentencePiece

1. **Fixed Vocabulary Size**
   - Must specify vocabulary size before training
   - May need tuning for optimal performance

2. **Training Time**
   - Can be slow for very large corpora
   - Unigram model is iterative

3. **Subword Ambiguity**
   - Same text may have different valid segmentations
   - Context-dependent tokenization can be complex

## Real-World Example: Loading a Pre-trained Model

In [ ]:
# Example: Using a pre-trained SentencePiece model (simulated)
# In practice, you would download models from HuggingFace or other sources

print("""
In real projects, you would load pre-trained models like this:

# From HuggingFace
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Or load SentencePiece directly
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load('path/to/model.model')

# Encode text
tokens = sp.encode("Hello world!", out_type='int')
print(tokens)
""")

## Summary

SentencePiece is a powerful, language-independent tokenization library that:

- Uses **Unigram LM** or **BPE** algorithms
- Requires **no preprocessing** (no need to split words by spaces)
- Provides **reversible tokenization**
- Supports **subword regularization** for better training
- Is used in many **state-of-the-art models** (ALBERT, T5, mBART)

It's particularly valuable for:
- **Multilingual models** requiring a single tokenizer
- **Languages without word boundaries** (Chinese, Japanese, Thai)
- **End-to-end pipelines** where detokenization is needed